# Dynamic GHZ State — Measure, Correct, and Reuse

**Companion notebook to the talk by Fatima Bint-e-Munir**  
Qiskit Fall Fest 2026 · University of Engineering & Technology Lahore, Narowal Campus · Talk 8

Build a genuine four-qubit GHZ state using one mid-circuit measurement, reset, and classical feedforward:
$$|\mathrm{GHZ}_4\rangle=\frac{|0000\rangle+|1111\rangle}{\sqrt{2}}.$$

We measure only the bridge $q_1$ during preparation. The edge qubit $q_3$ remains unmeasured until final readout. The last preparation gate is **`CX(0, 1)`**.

All kets and final output bitstrings use Qiskit's order **$q_3q_2q_1q_0$**. Run the cells from top to bottom. This notebook uses an ideal local simulator; no account or hardware access is needed.

## Setup
Install the packages once. In Colab or Jupyter, restart the kernel if requested after installation, then continue below.

In [ ]:
%pip install "qiskit>=2.0,<3.0" qiskit-aer matplotlib pylatexenc -q

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display
import qiskit

backend = AerSimulator()
shots = 4000
print("Qiskit version:", qiskit.__version__)

## Step 1 — Superpose the data qubits

Start all four qubits in $|0\rangle$. Apply $H$ to $q_0$ and $q_2$, creating two independent superpositions. The bridge $q_1$ and edge $q_3$ remain zero:
$$|\psi_1\rangle=\frac{|0000\rangle+|0001\rangle+|0100\rangle+|0101\rangle}{2}.$$

Use a dedicated classical bit `m[0]` for the bridge outcome. Final readout will use a separate register.

In [ ]:
qr = QuantumRegister(4, "q")
m = ClassicalRegister(1, "m")
qc = QuantumCircuit(qr, m)
qc.h(0)
qc.h(2)
qc.barrier()
display(qc.draw("mpl"))

## Step 2 — Create the first Bell pair

Apply `CX(2, 1)`: $q_2$ controls a flip of $q_1$. These two qubits form a Bell pair; $q_0$ remains independent:
$$|\psi_2\rangle=\frac{|0000\rangle+|0001\rangle+|0110\rangle+|0111\rangle}{2}.$$

In [ ]:
qc.cx(2, 1)
qc.barrier()
display(qc.draw("mpl"))

## Step 3 — Encode parity in the bridge

Apply `CX(0, 1)` and `CX(2, 3)`. The bridge now stores $q_0\oplus q_2$, while $q_3$ shares $q_2$'s value:
$$|\psi_3\rangle=\frac{|0000\rangle+|0011\rangle+|1110\rangle+|1101\rangle}{2}.$$

Measuring $q_1$ will reveal whether $q_0$ and $q_2$ agree, without revealing their individual values.

In [ ]:
qc.cx(0, 1)
qc.cx(2, 3)
qc.barrier()
display(qc.draw("mpl"))

## Step 4 — Measure the bridge, reset it, and correct

Measure $q_1$ into `m[0]`, then reset $q_1$ to $|0\rangle$.

- **$m_1=0$:** The data values agree. After reset, the state is $(|0000\rangle+|1101\rangle)/\sqrt{2}$. No correction is needed.
- **$m_1=1$:** The data values disagree. Immediately after measurement, the state is $(|0011\rangle+|1110\rangle)/\sqrt{2}$. Resetting $q_1$ gives $(|0001\rangle+|1100\rangle)/\sqrt{2}$. Conditional $X$ gates on both $q_2$ and $q_3$ transform this into the same state as the zero branch.

Therefore, both branches give
$$|\psi_4\rangle=\frac{|0000\rangle+|1101\rangle}{\sqrt{2}}.$$
This is a three-qubit GHZ state on $q_0,q_2,q_3$, with $q_1=0$. The measurement bit remains random; feedforward removes its effect on the prepared quantum state.

In [ ]:
qc.measure(qr[1], m[0])
qc.reset(qr[1])
with qc.if_test((m[0], 1)):
    qc.x(2)
    qc.x(3)
qc.barrier()
display(qc.draw("mpl"))

## Step 5 — Reuse the bridge to finish the GHZ state

Apply **only `CX(0, 1)`**. In the first component $q_0=0$, so nothing changes. In the second component $q_0=1$, so the reset bridge $q_1$ flips to one:
$$\frac{|0000\rangle+|1101\rangle}{\sqrt{2}}
\xrightarrow{CX(0,1)}
\frac{|0000\rangle+|1111\rangle}{\sqrt{2}}.$$

**Keep $q_3$ unmeasured during preparation:** it already belongs to the GHZ state. No additional `CX(2, 3)` is needed. The original six-step construction is now five preparation steps.

In [ ]:
qc.cx(0, 1)
qc.barrier()
qc_ghz = qc.copy()  # Preserve preparation before final measurement.
display(qc_ghz.draw("mpl"))

## Final readout — Check the Z-basis correlations

Measure all four qubits only after preparation. On the ideal simulator, only `0000` and `1111` should appear, approximately equally often.

The helper below reads only the four-bit register `c`; the separately stored bridge outcome `m` is excluded from the plotted bitstrings.

In [ ]:
def run_readout(preparation, x_basis=False):
    circuit = preparation.copy()
    readout = ClassicalRegister(4, "c")
    circuit.add_register(readout)
    if x_basis:
        circuit.h(range(4))
    circuit.measure(circuit.qubits, readout)
    compiled = transpile(circuit, backend)
    result = backend.run(compiled, shots=shots, seed_simulator=2026).result()
    joint_counts = result.get_counts()
    # Register c was added last, so it is the leftmost group in each key.
    counts = {}
    for key, count in joint_counts.items():
        final_bits = key.split()[0]
        counts[final_bits] = counts.get(final_bits, 0) + count
    return circuit, counts

qc_z, z_counts = run_readout(qc_ghz)
print("Z-basis counts:", z_counts)
display(qc_z.draw("mpl"))
display(plot_histogram(z_counts, title="Dynamic GHZ: Z-basis readout"))

## Check coherence — Measure in the X basis

Matching Z-basis outcomes alone cannot distinguish a GHZ superposition from a classical mixture. For an additional check, apply $H$ to each qubit immediately before final measurement. This measures the X basis.

For the prepared GHZ state, only bitstrings containing an **even number of ones** occur, giving $\langle X\otimes X\otimes X\otimes X\rangle=+1$. The equal classical mixture of `0000` and `1111` gives expectation zero instead. Together, ideal Z-basis support and this coherence check verify the intended GHZ state.

In [ ]:
qc_x, x_counts = run_readout(qc_ghz, x_basis=True)
xxxx = sum(count * (-1) ** bits.count("1")
           for bits, count in x_counts.items()) / sum(x_counts.values())
print("X-basis counts:", x_counts)
print(f"<XXXX> = {xxxx:+.3f} (ideal GHZ: +1.000)")
display(plot_histogram(x_counts, title="Dynamic GHZ: X-basis parity"))

## What this construction demonstrates

One random parity measurement, followed by reset and conditional correction, produces the same coherent state in either branch. A final CNOT reuses the bridge and completes the four-qubit GHZ state. Final readout is kept separate from preparation.